# Notebook 08: MLP training on bike rentals by hand

In this notebook, we will train a small multilayer perceptron (MLP) to predict bike-rental counts from a curated real dataset.

The goal is not to use a machine-learning library. The goal is to see the whole training path with plain Python lists and floats: data row -> prediction -> loss -> gradients -> parameter update.

We will start by loading one row, naming the five model inputs, and comparing the scaled target with the original rental count. Then we will build the forward pass, derive the backward pass by hand, and train with plain gradient descent.


## First example: one row becomes model inputs and a target

The CSV file has one bike-rental example per row. A **model input** is a number the model is allowed to use for its prediction. A **target** is the answer the model is trying to predict.

For the first row, we will use these five inputs: `hour_scaled`, `temperature`, `humidity`, `wind_speed`, and `working_day`. The scaled target is `target_scaled`, which is the rental count divided by `1000`.


In [33]:
import csv

from pathlib import Path

DATA_PATH = "../data/bike_rentals.csv"

rows = []

with Path(DATA_PATH).open(newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    column_names = reader.fieldnames

    for row in reader:
        rows.append(row)

first_row = rows[0]

input_values = [
    float(first_row["hour_scaled"]),
    float(first_row["temperature"]),
    float(first_row["humidity"]),
    float(first_row["wind_speed"]),
    float(first_row["working_day"]),
]

target_scaled = float(first_row["target_scaled"])

print("columns:", column_names)
print("number of rows:", len(rows))
print("inputs:", input_values)
print("target_scaled:", target_scaled)
print("original rental_count:", first_row["rental_count"])

columns: ['hour', 'hour_scaled', 'temperature', 'humidity', 'wind_speed', 'working_day', 'rental_count', 'target_scaled']
number of rows: 17379
inputs: [0.0, 0.24, 0.81, 0.0, 0.0]
target_scaled: 0.016
original rental_count: 16


## Model shape

Each training example gives the model five input numbers:

1. `hour_scaled`
2. `temperature`
3. `humidity`
4. `wind_speed`
5. `working_day`

The model will use this shape:

```text
5 inputs -> 6 ReLU hidden neurons -> 1 output
```

Each hidden neuron produces one hidden value. The output layer combines those hidden values into one scaled rental-count prediction.


## Forward pass as a reusable function

A forward pass sends one example through the model to produce one prediction. We will put the forward pass inside a function right away, because training needs to reuse the same calculation for many rows.

Inside the function, Layer 1 turns the five input values into hidden ReLU values. ReLU means “keep positive numbers, turn negative numbers into `0`.”

Layer 2 combines the hidden values into one scaled rental-count prediction. The function also returns a `ForwardCache` dataclass, which stores intermediate values the backward pass will need when it computes gradients.


In [34]:
from dataclasses import dataclass


def relu(value: float) -> float:
    return max(0.0, value)


@dataclass
class ForwardCache:
    input_values: list[float]
    hidden_raw_values: list[float]
    hidden_outputs: list[float]


hidden_size = 12

# Parameters for the input -> hidden layer
input_to_hidden_weights = [
    [0.10, -0.20, 0.05, 0.03, 0.08],
    [-0.04, 0.12, -0.07, 0.09, 0.02],
    [0.06, 0.01, 0.11, -0.05, -0.03],
    [-0.08, 0.04, 0.02, 0.10, 0.07],
    [0.03, -0.06, 0.09, 0.01, -0.04],
    [0.05, 0.07, -0.02, -0.08, 0.06],
]
hidden_biases = [0.01, -0.02, 0.00, 0.03, -0.01, 0.02]

# Parameters for the hidden -> output layer
hidden_to_output_weights = [0.20, -0.10, 0.15, 0.05, -0.08, 0.12]
output_bias = 0.01


def forward_pass(
    input_values: list[float],
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
) -> tuple[float, ForwardCache]:
    hidden_raw_values = []
    hidden_outputs = []

    # Layer 1: input values -> hidden ReLU values
    for hidden_index in range(len(hidden_biases)):
        raw_value = hidden_biases[hidden_index]

        for input_index in range(len(input_values)):
            raw_value += (
                input_values[input_index]
                * input_to_hidden_weights[hidden_index][input_index]
            )

        hidden_output = relu(raw_value)

        hidden_raw_values.append(raw_value)
        hidden_outputs.append(hidden_output)

    # Layer 2: hidden ReLU values -> one scaled prediction
    prediction = output_bias

    for hidden_index in range(len(hidden_outputs)):
        prediction += (
            hidden_outputs[hidden_index] * hidden_to_output_weights[hidden_index]
        )

    cache = ForwardCache(
        input_values=input_values,
        hidden_raw_values=hidden_raw_values,
        hidden_outputs=hidden_outputs,
    )

    return prediction, cache


prediction, cache = forward_pass(
    input_values,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)

print("prediction_scaled:", prediction)
print("target_scaled:", target_scaled)
print("cached hidden raw values:", cache.hidden_raw_values)
print("cached hidden outputs:", cache.hidden_outputs)

prediction_scaled: 0.025607
target_scaled: 0.016
cached hidden raw values: [0.002500000000000009, -0.04790000000000001, 0.09150000000000001, 0.0558, 0.04850000000000001, 0.020599999999999997]
cached hidden outputs: [0.002500000000000009, 0.0, 0.09150000000000001, 0.0558, 0.04850000000000001, 0.020599999999999997]


## Backward pass as one reusable function

The backward pass starts at the prediction error and computes every gradient we need for this one example.

It walks backward through the same path the forward pass used:

```text
loss -> prediction -> output layer -> hidden ReLU values -> input-to-hidden weights
```

The function returns a `BackwardPassResult` dataclass. That dataclass only keeps the update-ready parameter gradients: `hidden_to_output_weights`, `output_bias`, `input_to_hidden_weights`, and `hidden_biases`.

Intermediate gradients such as `d_loss_d_prediction` and `d_loss_d_hidden_outputs` still exist inside the function, but they are not returned because gradient descent will not update those values directly.


In [35]:
from dataclasses import dataclass


@dataclass
class BackwardPassResult:
    d_loss_d_hidden_to_output_weights: list[float]
    d_loss_d_output_bias: float
    d_loss_d_input_to_hidden_weights: list[list[float]]
    d_loss_d_hidden_biases: list[float]


def relu_derivative(raw_value: float) -> float:
    if raw_value > 0.0:
        return 1.0

    return 0.0


def backward_pass(
    prediction: float,
    target: float,
    cache: ForwardCache,
    hidden_to_output_weights: list[float],
) -> BackwardPassResult:
    input_values = cache.input_values
    hidden_raw_values = cache.hidden_raw_values
    hidden_outputs = cache.hidden_outputs

    # Start of backprop: loss -> prediction
    error = prediction - target
    d_loss_d_prediction = 2.0 * error

    d_loss_d_hidden_to_output_weights = []
    d_loss_d_hidden_outputs = []

    # Output layer: prediction -> hidden_to_output weights and hidden outputs
    for hidden_index, hidden_output in enumerate(hidden_outputs):
        # Compute the derivative of the loss with respect to this hidden-to-output weight
        d_prediction_d_hidden_to_output_weight = hidden_output
        d_loss_d_hidden_to_output_weight = (
            d_loss_d_prediction * d_prediction_d_hidden_to_output_weight
        )
        d_loss_d_hidden_to_output_weights.append(d_loss_d_hidden_to_output_weight)

        # Compute the derivative of the loss with respect to this hidden output after ReLU
        d_prediction_d_hidden_output = hidden_to_output_weights[hidden_index]
        d_loss_d_hidden_output = d_loss_d_prediction * d_prediction_d_hidden_output
        d_loss_d_hidden_outputs.append(d_loss_d_hidden_output)

    d_loss_d_output_bias = d_loss_d_prediction

    d_loss_d_input_to_hidden_weights = []
    d_loss_d_hidden_biases = []

    # Hidden layer: hidden ReLU values -> hidden raw values -> input weights and biases
    for hidden_index, hidden_raw_value in enumerate(hidden_raw_values):
        d_hidden_output_d_hidden_raw = relu_derivative(hidden_raw_value)
        d_loss_d_hidden_raw = (
            d_loss_d_hidden_outputs[hidden_index] * d_hidden_output_d_hidden_raw
        )

        hidden_weight_gradients = []

        for input_value in input_values:
            d_hidden_raw_d_input_to_hidden_weight = input_value
            d_loss_d_input_to_hidden_weight = (
                d_loss_d_hidden_raw * d_hidden_raw_d_input_to_hidden_weight
            )
            hidden_weight_gradients.append(d_loss_d_input_to_hidden_weight)

        d_loss_d_input_to_hidden_weights.append(hidden_weight_gradients)

        d_loss_d_hidden_bias = d_loss_d_hidden_raw
        d_loss_d_hidden_biases.append(d_loss_d_hidden_bias)

    return BackwardPassResult(
        d_loss_d_hidden_to_output_weights=d_loss_d_hidden_to_output_weights,
        d_loss_d_output_bias=d_loss_d_output_bias,
        d_loss_d_input_to_hidden_weights=d_loss_d_input_to_hidden_weights,
        d_loss_d_hidden_biases=d_loss_d_hidden_biases,
    )


gradients = backward_pass(
    prediction,
    target_scaled,
    cache,
    hidden_to_output_weights,
)

current_error = prediction - target_scaled
current_loss = current_error * current_error

print("loss:", current_loss)
print("d_loss_d_hidden_to_output_weights:", gradients.d_loss_d_hidden_to_output_weights)
print("d_loss_d_output_bias:", gradients.d_loss_d_output_bias)
print(
    "d_loss_d_input_to_hidden_weights[0]:",
    gradients.d_loss_d_input_to_hidden_weights[0],
)
print("d_loss_d_hidden_biases:", gradients.d_loss_d_hidden_biases)

loss: 9.229444900000002e-05
d_loss_d_hidden_to_output_weights: [4.803500000000018e-05, 0.0, 0.0017580810000000003, 0.0010721412, 0.0009318790000000003, 0.00039580839999999996]
d_loss_d_output_bias: 0.019214000000000002
d_loss_d_input_to_hidden_weights[0]: [0.0, 0.000922272, 0.0031126680000000007, 0.0, 0.0]
d_loss_d_hidden_biases: [0.0038428000000000004, -0.0, 0.0028821000000000003, 0.0009607000000000001, -0.0015371200000000001, 0.00230568]


## Training loop on all rows

A training loop repeats the same learning steps many times:

```text
forward pass -> backward pass -> average gradients -> parameter update
```

Now we will train on every row loaded from the full bike-rental CSV. For each epoch, we compute gradients for every row, average those gradients, and then apply one parameter update.

Averaging keeps the update size consistent when the number of training examples changes.


In [36]:
TrainingExample = tuple[list[float], float]


def row_to_training_example(row: dict[str, str]) -> TrainingExample:
    input_values = [
        float(row["hour_scaled"]),
        float(row["temperature"]),
        float(row["humidity"]),
        float(row["wind_speed"]),
        float(row["working_day"]),
    ]
    target_scaled = float(row["target_scaled"])

    return input_values, target_scaled


def make_initial_parameters() -> tuple[
    list[list[float]],
    list[float],
    list[float],
    float,
]:
    return (
        [
            [0.10, -0.20, 0.05, 0.03, 0.08],
            [-0.04, 0.12, -0.07, 0.09, 0.02],
            [0.06, 0.01, 0.11, -0.05, -0.03],
            [-0.08, 0.04, 0.02, 0.10, 0.07],
            [0.03, -0.06, 0.09, 0.01, -0.04],
            [0.05, 0.07, -0.02, -0.08, 0.06],
        ],
        [0.01, -0.02, 0.00, 0.03, -0.01, 0.02],
        [0.20, -0.10, 0.15, 0.05, -0.08, 0.12],
        0.01,
    )


def zero_gradients(hidden_size: int, input_size: int) -> BackwardPassResult:
    return BackwardPassResult(
        d_loss_d_hidden_to_output_weights=[0.0 for _ in range(hidden_size)],
        d_loss_d_output_bias=0.0,
        d_loss_d_input_to_hidden_weights=[
            [0.0 for _ in range(input_size)] for _ in range(hidden_size)
        ],
        d_loss_d_hidden_biases=[0.0 for _ in range(hidden_size)],
    )


def add_gradients_in_place(
    total_gradients: BackwardPassResult,
    gradients: BackwardPassResult,
) -> None:
    for hidden_index, weight_gradient in enumerate(
        gradients.d_loss_d_hidden_to_output_weights
    ):
        total_gradients.d_loss_d_hidden_to_output_weights[hidden_index] += (
            weight_gradient
        )

    total_gradients.d_loss_d_output_bias += gradients.d_loss_d_output_bias

    for hidden_index, hidden_weight_gradients in enumerate(
        gradients.d_loss_d_input_to_hidden_weights
    ):
        for input_index, weight_gradient in enumerate(hidden_weight_gradients):
            total_gradients.d_loss_d_input_to_hidden_weights[hidden_index][
                input_index
            ] += weight_gradient

    for hidden_index, bias_gradient in enumerate(gradients.d_loss_d_hidden_biases):
        total_gradients.d_loss_d_hidden_biases[hidden_index] += bias_gradient


def scale_gradients(
    gradients: BackwardPassResult,
    scale: float,
) -> BackwardPassResult:
    return BackwardPassResult(
        d_loss_d_hidden_to_output_weights=[
            gradient * scale
            for gradient in gradients.d_loss_d_hidden_to_output_weights
        ],
        d_loss_d_output_bias=gradients.d_loss_d_output_bias * scale,
        d_loss_d_input_to_hidden_weights=[
            [gradient * scale for gradient in hidden_weight_gradients]
            for hidden_weight_gradients in gradients.d_loss_d_input_to_hidden_weights
        ],
        d_loss_d_hidden_biases=[
            gradient * scale for gradient in gradients.d_loss_d_hidden_biases
        ],
    )


def apply_gradients(
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
    gradients: BackwardPassResult,
    learning_rate: float,
) -> float:
    # Update hidden -> output weights
    for hidden_index, weight_gradient in enumerate(
        gradients.d_loss_d_hidden_to_output_weights
    ):
        hidden_to_output_weights[hidden_index] -= learning_rate * weight_gradient

    # Update output bias
    output_bias -= learning_rate * gradients.d_loss_d_output_bias

    # Update input -> hidden weights
    for hidden_index, hidden_weight_gradients in enumerate(
        gradients.d_loss_d_input_to_hidden_weights
    ):
        for input_index, weight_gradient in enumerate(hidden_weight_gradients):
            input_to_hidden_weights[hidden_index][input_index] -= (
                learning_rate * weight_gradient
            )

    # Update hidden biases
    for hidden_index, bias_gradient in enumerate(gradients.d_loss_d_hidden_biases):
        hidden_biases[hidden_index] -= learning_rate * bias_gradient

    return output_bias


def mean_squared_error(
    training_examples: list[TrainingExample],
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
) -> float:
    total_loss = 0.0

    for example_inputs, example_target in training_examples:
        prediction, _ = forward_pass(
            example_inputs,
            input_to_hidden_weights,
            hidden_biases,
            hidden_to_output_weights,
            output_bias,
        )
        error = prediction - example_target
        total_loss += error * error

    return total_loss / len(training_examples)


training_examples = [row_to_training_example(row) for row in rows]

(
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
) = make_initial_parameters()

learning_rate = 0.1
epochs = 250

initial_loss = mean_squared_error(
    training_examples,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)
print(f"initial training loss: {initial_loss:.6f}")

for epoch in range(epochs):
    total_gradients = zero_gradients(
        hidden_size=len(hidden_biases),
        input_size=len(input_to_hidden_weights[0]),
    )
    total_loss = 0.0

    for example_inputs, example_target in training_examples:
        prediction, cache = forward_pass(
            example_inputs,
            input_to_hidden_weights,
            hidden_biases,
            hidden_to_output_weights,
            output_bias,
        )
        error = prediction - example_target
        total_loss += error * error

        gradients = backward_pass(
            prediction,
            example_target,
            cache,
            hidden_to_output_weights,
        )
        add_gradients_in_place(total_gradients, gradients)

    average_loss = total_loss / len(training_examples)
    average_gradients = scale_gradients(
        total_gradients,
        1.0 / len(training_examples),
    )

    output_bias = apply_gradients(
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
        average_gradients,
        learning_rate,
    )

    if epoch == 0 or (epoch + 1) % 50 == 0:
        print(f"epoch: {epoch + 1} training loss: {average_loss:.6f}")

final_loss = mean_squared_error(
    training_examples,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)

print(f"final training loss: {final_loss:.6f}")

for row_index, (example_inputs, example_target) in enumerate(
    training_examples[:5],
):
    prediction, _ = forward_pass(
        example_inputs,
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
    )
    predicted_count = prediction * 1000.0
    actual_count = example_target * 1000.0

    print(
        f"row: {row_index + 1} "
        f"predicted rentals: {predicted_count:.1f} "
        f"actual rentals: {actual_count:.1f}"
    )


initial training loss: 0.053707
epoch: 1 training loss: 0.053707
epoch: 50 training loss: 0.031440
epoch: 100 training loss: 0.029959
epoch: 150 training loss: 0.028428
epoch: 200 training loss: 0.026925
epoch: 250 training loss: 0.025602
epoch: 300 training loss: 0.024570
epoch: 350 training loss: 0.023830
epoch: 400 training loss: 0.023318
epoch: 450 training loss: 0.022964
epoch: 500 training loss: 0.022716
final training loss: 0.022712
row: 1 predicted rentals: 23.9 actual rentals: 16.0
row: 2 predicted rentals: 25.1 actual rentals: 40.0
row: 3 predicted rentals: 27.9 actual rentals: 32.0
row: 4 predicted rentals: 35.8 actual rentals: 13.0
row: 5 predicted rentals: 41.2 actual rentals: 1.0


In [37]:
# For regression, "accuracy" needs a tolerance.
# Here we report average error and the percentage of predictions
# that land within 50 and 100 rentals of the actual count.

total_absolute_error = 0.0
total_squared_error = 0.0
within_50_rentals = 0
within_100_rentals = 0

for example_inputs, example_target in training_examples:
    prediction, _ = forward_pass(
        example_inputs,
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
    )

    predicted_count = prediction * 1000.0
    actual_count = example_target * 1000.0
    absolute_error = abs(predicted_count - actual_count)

    total_absolute_error += absolute_error
    total_squared_error += absolute_error * absolute_error

    if absolute_error <= 50.0:
        within_50_rentals += 1

    if absolute_error <= 100.0:
        within_100_rentals += 1

example_count = len(training_examples)
mean_absolute_error = total_absolute_error / example_count
root_mean_squared_error = (total_squared_error / example_count) ** 0.5
within_50_accuracy = within_50_rentals / example_count * 100.0
within_100_accuracy = within_100_rentals / example_count * 100.0

print(f"mean absolute error: {mean_absolute_error:.1f} rentals")
print(f"root mean squared error: {root_mean_squared_error:.1f} rentals")
print(f"within 50 rentals: {within_50_accuracy:.1f}%")
print(f"within 100 rentals: {within_100_accuracy:.1f}%")


mean absolute error: 110.3 rentals
root mean squared error: 150.7 rentals
within 50 rentals: 30.9%
within 100 rentals: 58.2%
